In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                       "huggingface_hub<1.0", "hf-transfer"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "git+https://github.com/mahmoodlab/CONCH.git"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
DATASET = "pathmnist"  # pathmnist | histoset | skintissue
SEED = 42  # any int; one seed per run

VLM = "MahmoodLab/CONCH"  # only CONCH is verified; others need a loader in features/vlm.py

# The one LIST axis here: image features never see a prompt, so every style
# reuses the same 448x448 pass and costs 14 strings each.
DESCRIPTION_STYLES = ["llm_short", "llm_morphology"]  # llm_short | llm_morphology

HF_TOKEN = ""  # "" = fall back to Kaggle Secret HF_TOKEN

DATA_ROOT = "/kaggle/input/datasets/cryandrrich/nckh2026"
FEATURE_DIR = "/kaggle/working/vlm_features"

# A cache this notebook published earlier, attached via Add Data. Point at it to
# add a style without re-extracting the image features. "" = nothing to reuse.
PREVIOUS_VLM_DIR = "/kaggle/input/vlm-histoset-seed42-mahmoodlab-conch-llm-short"

PARALLEL = True  # True | False

BATCH_SIZE = 64  # 64 tested on a 16 GiB T4; lower on CUDA OOM

MMAP_CACHE_DIR = "/kaggle/working/npz_mmap"


In [ ]:
from huggingface_hub import login

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient

        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print("[auth] using Kaggle Secret HF_TOKEN")
    except Exception as exc:
        print(f"[auth] no Kaggle Secret HF_TOKEN ({type(exc).__name__})")

assert HF_TOKEN, (
    "CONCH is gated on Hugging Face and this notebook downloads it, so a "
    "token is required. Either set HF_TOKEN in the EDIT cell, or add a "
    "Kaggle Secret named HF_TOKEN (Add-ons -> Secrets). Also accept the "
    "license once at https://huggingface.co/MahmoodLab/CONCH -- a valid "
    "token for an account that has not accepted still gets 401/403."
)

login(HF_TOKEN)
import os

os.environ["HF_TOKEN"] = HF_TOKEN
print("[auth] logged in to the HF Hub; CONCH checkpoint will download on first use")

In [ ]:
import json

import numpy as np
import yaml
import torch

from data.loaders import get_data_loaders, get_sample_ids
from data.identity import sample_order_fingerprint
from data.npz_mmap import export_npz_to_npy
from features.vlm import (
    RAW_SPACE,
    PROJ_SPACE,
    assemble_vlm_feature_shards,
    description_sha256,
    encode_text_prototypes,
    get_or_extract_vlm_features,
    load_conch,
    text_prototype_cache_paths,
    vlm_feature_cache_paths,
    zero_shot_logits,
)
from scripts.extract_vlm_features import build_vlm_shard_jobs, extract_vlm_shard_on_worker
from utils import vlm_archive_stem
from utils.kaggle import find_data_root, find_vlm_cache
from utils.parallel import run_variants_parallel, visible_gpu_count

import main as _main

In [ ]:
DATA_ROOT_RESOLVED = find_data_root([Path(DATA_ROOT)])

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT_RESOLVED / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT_RESOLVED / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT_RESOLVED / "SkinTissue/SkinTissue/tiles"),
}
print("data root (raw images):", DATA_ROOT_RESOLVED)

In [ ]:
with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

assert isinstance(SEED, int), "SEED is one seed, not a list -- re-run the notebook to sweep"
data_path = Path(DATA_PATHS[DATASET])
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert torch.cuda.is_available(), "Attach a Kaggle GPU before extraction"
assert not str(FEATURE_DIR).startswith("/kaggle/input"), (
    "FEATURE_DIR must be writable; /kaggle/input is read-only"
)
Path(FEATURE_DIR).mkdir(parents=True, exist_ok=True)

# Reuse an earlier cache: image features do not depend on the style, so adding
# one costs 14 strings instead of a 448x448 pass. Copied, not read in place --
# /kaggle/input is read-only and new prototypes are written beside them.
mounted_vlm = find_vlm_cache(DATASET, SEED, VLM, hint=PREVIOUS_VLM_DIR)
if mounted_vlm is not None and Path(mounted_vlm).resolve() != Path(FEATURE_DIR).resolve():
    import shutil as _shutil

    copied = []
    for source in sorted(Path(mounted_vlm).glob(f"{DATASET}_*")):
        if source.is_file():
            target = Path(FEATURE_DIR) / source.name
            if not target.exists():
                _shutil.copy2(source, target)
                copied.append(source.name)
    print(f"[cache] reusing VLM cache from {mounted_vlm}")
    for name in copied:
        print(f"        copied {name}")
    if not copied:
        print("        (already present in FEATURE_DIR)")
else:
    print(f"[cache] no previous VLM cache found under {PREVIOUS_VLM_DIR!r} -- "
          "image features will be extracted this session")

dataset_info = config["datasets"][DATASET]
class_names = list(dataset_info["class_names"])
num_classes = dataset_info["num_classes"]
assert len(class_names) == num_classes, (
    f"config.yaml lists {len(class_names)} class_names but "
    f"num_classes={num_classes} for {DATASET!r}"
)

def resolve_style(style):
    """Return `(effective_style, class_prompts, source, sha256)` for one style.

    `effective_style` differs from `style` only for the fallback: a missing
    description file falls back to bare class names and renames itself, so a
    fallback run can never overwrite a real one.
    """
    description_path = Path(f"config/descriptions/{DATASET}_{style}.json")
    if description_path.is_file():
        with open(description_path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
        assert list(payload["descriptions"]) == class_names, (
            f"{description_path} class order does not match config.yaml's -- "
            "regenerate it against the current config"
        )
        return (
            style,
            [[payload["descriptions"][name]] for name in class_names],
            str(description_path),
            payload.get("sha256") or description_sha256(payload["descriptions"]),
        )

    classname_prompts = {name: name.replace("_", " ") for name in class_names}
    effective = f"{style}_fallback_classname"
    print(
        f"[descriptions] {description_path} not found -- falling back to bare "
        f"class names, tagged {effective!r}. It will NOT overwrite a real "
        f"{style!r} cache. Re-run after generate_class_description.ipynb."
    )
    return (
        effective,
        [[classname_prompts[name]] for name in class_names],
        f"class names (fallback: {description_path} not found)",
        description_sha256(classname_prompts),
    )


# Resolved before any GPU work, so a bad style name or a stale class order
# fails here rather than after the image extraction is paid for.
assert DESCRIPTION_STYLES, "DESCRIPTION_STYLES needs at least one style"
STYLE_PLAN = [resolve_style(style) for style in DESCRIPTION_STYLES]
EFFECTIVE_STYLES = [effective for effective, _, _, _ in STYLE_PLAN]
assert len(set(EFFECTIVE_STYLES)) == len(EFFECTIVE_STYLES), (
    f"two styles resolved to the same name: {EFFECTIVE_STYLES}"
)

print(f"VLM: {VLM} | styles: {EFFECTIVE_STYLES}")
for effective, prompts, source, _ in STYLE_PLAN:
    print(f"  {effective:28} <- {source}")
print(f"classes ({len(class_names)}): {class_names}")
SHARDS = visible_gpu_count() if PARALLEL else 1
SHARDS = max(1, SHARDS)

import shutil as _shutil

total_npz = data_path.stat().st_size if str(data_path).endswith(".npz") else 0
free_disk = _shutil.disk_usage("/kaggle/working").free if Path("/kaggle/working").exists() \
    else _shutil.disk_usage(".").free
print(f"GPUs visible: {visible_gpu_count()} | shards: {SHARDS} | batch: {BATCH_SIZE}")
if total_npz:
    print(f"npz input: {total_npz / 2**30:.1f} GiB | mmap export needs about the same")
    print(f"free disk: {free_disk / 2**30:.1f} GiB at {MMAP_CACHE_DIR}")
    assert free_disk > total_npz * 1.1, (
        f"Not enough scratch disk for the .npy export: need ~{total_npz / 2**30:.1f} GiB, "
        f"have {free_disk / 2**30:.1f} GiB."
    )
try:
    with open("/proc/meminfo") as handle:
        mem_total = int(next(l for l in handle if l.startswith("MemTotal")).split()[1]) * 1024
    print(f"system RAM: {mem_total / 2**30:.1f} GiB across {SHARDS} worker(s)")
except (OSError, StopIteration, ValueError):
    pass

In [ ]:
import time

started = time.time()
device = torch.device("cuda:0")
print(f"{DATASET} | seed {SEED} | {VLM} | shards={SHARDS}")

vlm_paths = vlm_feature_cache_paths(FEATURE_DIR, DATASET, SEED, VLM)

mmap_dir = None
if str(data_path).endswith(".npz"):
    export_npz_to_npy(str(data_path), MMAP_CACHE_DIR)
    mmap_dir = MMAP_CACHE_DIR
    print(f"mmap export ready: {MMAP_CACHE_DIR}")
else:
    print("ImageFolder dataset: reads per file, no mmap export needed")

train_loader, test_loader, _ = get_data_loaders(
    str(data_path), SEED, verbose=True, mmap_cache_dir=mmap_dir,
)
n_train, n_test = len(train_loader.dataset), len(test_loader.dataset)
train_fingerprint = sample_order_fingerprint(get_sample_ids(train_loader.dataset))
test_fingerprint = sample_order_fingerprint(get_sample_ids(test_loader.dataset))
test_labels = _main._dataset_labels(test_loader.dataset)
print(f"  train={n_train} test={n_test} mmap={getattr(train_loader.dataset, 'mmap', None)}")

del train_loader, test_loader

import time as _time

_pf_started = _time.time()
_pf_device = torch.device("cuda:0")
_pf_model, _pf_preprocess = load_conch(VLM, _pf_device, hf_token=HF_TOKEN)
print(f"[preflight] CONCH loaded in {_time.time() - _pf_started:.0f}s")

_pf_t0 = _time.time()
# One style is enough here: this proves the text tower and tokenizer
# work at all, which is the same code path for every style.
_pf_style, _pf_prompts = STYLE_PLAN[0][0], STYLE_PLAN[0][1]
_pf_text = encode_text_prototypes(_pf_model, class_names, _pf_prompts, _pf_device)
print(
    f"[preflight] text prototypes {tuple(_pf_text.shape)} in "
    f"{_time.time() - _pf_t0:.1f}s  (style={_pf_style}, of {len(STYLE_PLAN)} planned)"
)
assert _pf_text.shape[0] == num_classes, (
    f"expected one prototype per class, got {_pf_text.shape}"
)

_pf_loader, _, _ = get_data_loaders(
    str(data_path), SEED, mmap_cache_dir=mmap_dir, transform=_pf_preprocess,
)
_pf_batch_loader = torch.utils.data.DataLoader(
    _pf_loader.dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=_pf_loader.num_workers, pin_memory=True,
)
_pf_iter = iter(_pf_batch_loader)
_pf_images = next(_pf_iter)[0]
print(f"[preflight] batch {tuple(_pf_images.shape)} via CONCH preprocess")
assert _pf_images.shape[-2:] == (448, 448), (
    f"CONCH expects 448x448, got {tuple(_pf_images.shape[-2:])} -- wrong transform"
)

with torch.inference_mode():
    _pf_model.visual.forward_no_head(_pf_images.to(_pf_device))
    torch.cuda.synchronize()
    _pf_t0 = _time.time()
    _pf_raw = _pf_model.visual.forward_no_head(_pf_images.to(_pf_device))
    torch.cuda.synchronize()
    _pf_per_batch = _time.time() - _pf_t0

_pf_n = _pf_images.shape[0]
_pf_total_images = n_train + n_test
_pf_gpu_hours = _pf_per_batch / _pf_n * _pf_total_images / 3600
_pf_wall_hours = _pf_gpu_hours / SHARDS

print(f"[preflight] raw space {tuple(_pf_raw.shape)}  "
      f"({_pf_per_batch / _pf_n * 1000:.0f} ms/image, {_pf_n / _pf_per_batch:.0f} img/s on 1 GPU)")
print(f"[preflight] VRAM peak {torch.cuda.max_memory_allocated(_pf_device) / 2**30:.1f} GiB "
      f"at batch {BATCH_SIZE}")
print(f"[preflight] {_pf_total_images} images "
      f"({n_train} train + {n_test} test)")
print(f"[preflight] ESTIMATE: {_pf_gpu_hours:.1f} GPU-hours "
      f"-> ~{_pf_wall_hours:.1f} h wall clock across {SHARDS} GPU(s)")
if _pf_wall_hours > 11.0:
    print(f"[preflight] WARNING: {_pf_wall_hours:.1f} h is close to Kaggle's 12 h "
          f"session cap. Consider a smaller dataset or more shards.")
print("[preflight] the pilot measures a warmed batch and ignores data loading, "
      "so treat this as a LOWER bound on wall clock.")

del _pf_raw, _pf_images, _pf_iter, _pf_batch_loader, _pf_loader, _pf_text
del _pf_model, _pf_preprocess
import gc; gc.collect()
torch.cuda.empty_cache()
print(f"[preflight] done in {_time.time() - _pf_started:.0f}s, GPU released")

if SHARDS == 1:
    conch_model, conch_preprocess = load_conch(VLM, device, hf_token=HF_TOKEN)
    train_loader, test_loader, _ = get_data_loaders(
        str(data_path), SEED, mmap_cache_dir=mmap_dir, transform=conch_preprocess,
    )
    train_loader = torch.utils.data.DataLoader(
        train_loader.dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=train_loader.num_workers, pin_memory=True,
    )
    test_loader = torch.utils.data.DataLoader(
        test_loader.dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=test_loader.num_workers, pin_memory=True,
    )
    cached = get_or_extract_vlm_features(
        train_loader, test_loader, DATASET, SEED, VLM, device,
        cache_dir=FEATURE_DIR,
        train_fingerprint=train_fingerprint, test_fingerprint=test_fingerprint,
        model=conch_model, hf_token=HF_TOKEN,
    )
    del train_loader, test_loader
elif Path(vlm_paths["manifest"]).is_file() and Path(vlm_paths["proj_manifest"]).is_file():
    print("complete cache exists, skipping extraction")
    cached = {k: np.load(vlm_paths[k]) for k in ("train", "test", "proj_train", "proj_test")}
    conch_model, _ = load_conch(VLM, device, hf_token=HF_TOKEN)
else:
    jobs = build_vlm_shard_jobs(
        str(data_path), DATASET, SEED, VLM, SHARDS, FEATURE_DIR,
        batch_size=BATCH_SIZE, mmap_cache_dir=mmap_dir, hf_token=HF_TOKEN or None,
    )
    results = run_variants_parallel(jobs, extract_vlm_shard_on_worker, num_workers=SHARDS)
    for result in results:
        status = "ok" if result["ok"] else "FAILED"
        print(f"  {result['label']:36} {status}")
    failed = [r["label"] for r in results if not r["ok"]]
    assert not failed, f"shards failed: {failed}"

    cached = assemble_vlm_feature_shards(
        DATASET, SEED, VLM, SHARDS, n_train, n_test, cache_dir=FEATURE_DIR,
        train_fingerprint=train_fingerprint, test_fingerprint=test_fingerprint,
    )
    conch_model, _ = load_conch(VLM, device, hf_token=HF_TOKEN)

print(f"raw:  train {cached['train'].shape} | test {cached['test'].shape}")
print(f"proj: train {cached['proj_train'].shape} | test {cached['proj_test'].shape}")
print(f"total {time.time() - started:.0f}s")

In [ ]:
# Every style's text prototypes, reusing the ONE loaded model.
text_prototypes_by_style = {}

for style, class_prompts, description_source, description_hash in STYLE_PLAN:
    text_paths = text_prototype_cache_paths(FEATURE_DIR, DATASET, style)

    prototypes_match = False
    if Path(text_paths["prototypes"]).is_file() and Path(text_paths["manifest"]).is_file():
        with open(text_paths["manifest"], "r", encoding="utf-8") as handle:
            text_manifest = json.load(handle)
        # The description file can change under a style name; the hash notices.
        prototypes_match = (
            text_manifest.get("class_names") == class_names
            and text_manifest.get("description_sha256") == description_hash
            # REQUIRED: the AL text prior reads it from here and it changes the
            # ORDER of the weights. Without this check a manifest written before
            # it existed is reused and the gap survives a re-run.
            and "logit_scale" in text_manifest
        )
        if prototypes_match:
            text_prototypes_by_style[style] = np.load(text_paths["prototypes"])
            print(f"[text] {style:28} loaded cache "
                  f"{text_prototypes_by_style[style].shape}")
        else:
            print(f"[text] {style:28} cache stale (class order or description "
                  "changed) -- recomputing")

    if not prototypes_match:
        prototypes_t = encode_text_prototypes(conch_model, class_names, class_prompts, device)
        prototypes = prototypes_t.cpu().numpy().astype(np.float32)
        os.makedirs(FEATURE_DIR, exist_ok=True)
        np.save(text_paths["prototypes"], prototypes)
        with open(text_paths["manifest"], "w", encoding="utf-8") as handle:
            json.dump({
                "dataset": DATASET,
                "style": style,
                "vlm": VLM,
                "class_names": class_names,
                "description_source": description_source,
                "description_sha256": description_hash,
                "prompts_per_class": [len(p) for p in class_prompts],
                # The AL text prior needs this and cannot recover it: an AL run
                # never loads CONCH. It changes the margin's top-2 gap.
                "logit_scale": float(conch_model.logit_scale.exp().item()),
            }, handle, indent=2, sort_keys=True)
        text_prototypes_by_style[style] = prototypes
        print(f"[text] {style:28} saved {prototypes.shape} -> "
              f"{Path(text_paths['prototypes']).name}")

assert set(text_prototypes_by_style) == set(EFFECTIVE_STYLES), (
    f"built {sorted(text_prototypes_by_style)} but planned {sorted(EFFECTIVE_STYLES)}"
)

# A property of the model, not the prompts -- shared by every style.
conch_logit_scale = conch_model.logit_scale.exp().item()

del conch_model
if device.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# Zero-shot on the test split: REPORTED, never asserted.
#
# It exercises the whole chain -- transform, tokenizer, projection head,
# logit_scale, and the class ORDER shared by prototypes and labels -- so a
# score at chance means something upstream is wrong. But it does not stop the
# notebook: the extraction is already paid for by this point, and throwing it
# away over a number is worse than printing the number and letting a human
# decide. Read the lines below before publishing the cache.
from scipy.stats import binomtest

labels = np.asarray(test_labels)
chance = 1.0 / num_classes
zero_shot_by_style = {}
suspect = []

for style, prototypes in text_prototypes_by_style.items():
    probs = zero_shot_logits(cached["proj_test"], prototypes, logit_scale=conch_logit_scale)
    predictions = probs.argmax(axis=1)
    accuracy = float((predictions == labels).mean())
    zero_shot_by_style[style] = accuracy

    correct = int((predictions == labels).sum())
    p_value = binomtest(correct, len(labels), chance, alternative="greater").pvalue
    predicted_classes = len(np.unique(predictions))

    print(f"[zero-shot] {style:28} acc {accuracy:.4f}  "
          f"(chance {chance:.3f}, p={p_value:.2e}, "
          f"{predicted_classes}/{num_classes} classes predicted)")

    if p_value >= 1e-6:
        suspect.append(f"{style}: indistinguishable from chance (p={p_value:.2e})")
    if predicted_classes <= 1:
        suspect.append(f"{style}: every image assigned one class")

if suspect:
    # What CANNOT be the cause, so this does not send anyone down a dead end:
    # logit_scale is irrelevant (argmax(softmax(x*s)) == argmax(x) for any
    # positive s -- measured, 0 of 500 predictions change), and so is
    # L2-normalizing the IMAGE rows (0.0% change), because both scale a whole
    # row that every class shares. What does move predictions: text prototypes
    # not unit-norm (~1.5%), and class ORDER mismatch (~22% for one swapped pair).
    print("\n[WARNING] zero-shot looks broken, not merely hard:")
    for line in suspect:
        print(f"    {line}")
    print("  In order of likelihood:")
    print("    1. class ORDER: config.yaml `class_names` and the dataset's")
    print("       label indices must agree. One swapped pair moves ~22%.")
    print("    2. the image transform: must be the `preprocess` object")
    print("       load_conch returned, not a hand-rolled 448 resize.")
    print("    3. the description file: check it is the right dataset's.")
    print("  NOT worth checking: logit_scale, or whether image rows are")
    print("  L2-normalized -- neither can change an argmax.")
    print("  The cache is still written; decide whether to publish it.")

# Reported in the summary below. Not a winner: comparing prompt styles is the
# EXPERIMENT, not this notebook's job.
zero_shot_accuracy = zero_shot_by_style[EFFECTIVE_STYLES[0]]


In [ ]:
vlm_paths = vlm_feature_cache_paths(FEATURE_DIR, DATASET, SEED, VLM)
for split in ("train", "test", "proj_train", "proj_test"):
    array = np.load(vlm_paths[split], mmap_mode="r")
    assert np.all(np.isfinite(array[:256])), f"{split} features are not all finite"
with open(vlm_paths["manifest"], "r", encoding="utf-8") as handle:
    manifest = json.load(handle)
with open(vlm_paths["proj_manifest"], "r", encoding="utf-8") as handle:
    proj_manifest = json.load(handle)
assert manifest["space"] == RAW_SPACE and proj_manifest["space"] == PROJ_SPACE
assert manifest["dataset"] == DATASET and manifest["seed"] == SEED
assert manifest["backbone"] == VLM

# Every style's prototypes, checked the same way.
for style, prototypes in text_prototypes_by_style.items():
    style_paths = text_prototype_cache_paths(FEATURE_DIR, DATASET, style)
    assert Path(style_paths["prototypes"]).is_file(), style
    assert Path(style_paths["manifest"]).is_file(), style
    # Against the IMAGE width, not against itself: `(num_classes, x.shape[1])`
    # holds for any width and would only check the row count.
    assert prototypes.shape == (num_classes, cached["proj_test"].shape[1]), (
        f"{style}: prototypes are {prototypes.shape} but PROJ_SPACE image "
        f"features are {cached['proj_test'].shape} -- these are dot-producted"
    )
    # Degenerate prototypes still have the right shape.
    norms = np.linalg.norm(prototypes, axis=1)
    assert np.all(norms > 0.9), f"{style}: prototypes are not unit-norm: {norms}"

print(f"OK {DATASET} seed{SEED} {VLM}")
print(f"   image  {cached['train'].shape} raw / {cached['proj_train'].shape} proj")
for style, accuracy in zero_shot_by_style.items():
    print(f"   text   {style:28} {text_prototypes_by_style[style].shape}  "
          f"zero-shot {accuracy:.4f}")


In [ ]:
import shutil

if MMAP_CACHE_DIR and Path(MMAP_CACHE_DIR).is_dir():
    freed = sum(f.stat().st_size for f in Path(MMAP_CACHE_DIR).rglob("*") if f.is_file())
    shutil.rmtree(MMAP_CACHE_DIR, ignore_errors=True)
    print(f"removed mmap export, freed {freed / 2**30:.1f} GiB")
SOURCE = Path(FEATURE_DIR)
WORKING = Path("/kaggle/working")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
assert SOURCE.resolve() != WORKING.resolve(), (
    "FEATURE_DIR must be a subdirectory of /kaggle/working, not /kaggle/working itself"
)

# Every style built in this run is named in the archive, so what is
# inside is readable off the filename without opening it.
STEM = vlm_archive_stem(DATASET, SEED, VLM, EFFECTIVE_STYLES)
ARCHIVE = WORKING / STEM
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6

print(f"{ARCHIVE.name}.zip  ({size_mb:.1f} MB) contains:")
for path in sorted(SOURCE.iterdir()):
    print(f"    {path.name}  ({path.stat().st_size / 1e6:.2f} MB)")

shutil.rmtree(SOURCE, ignore_errors=True)

remaining = sorted(p for p in WORKING.iterdir() if p.name != "codapath")
total_mb = sum(
    f.stat().st_size for p in remaining for f in ([p] if p.is_file() else p.rglob("*"))
    if f.is_file()
) / 1e6
print(f"\n/kaggle/working now holds {total_mb:.1f} MB (Output quota ~20 GB):")
for path in remaining:
    print(f"    {path.name}{'/' if path.is_dir() else ''}")

print(f"""
NEXT STEPS (no terminal needed)
  1. Output tab (right panel) -> download {ARCHIVE.name}.zip
  2. kaggle.com/datasets -> New Dataset -> upload that zip
     Kaggle extracts it into a directory named after the zip, so the cache
     files end up one level down. That is expected.
  3. In run_al_main.ipynb: Add Data -> your new dataset. The VLM feature
     cache is resolved by filename, so there is no path to edit.""")